In [8]:
import numpy as np 
import time

import torch 
import torch.nn as nn
import torch.nn.functional as F 
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
from collections import OrderedDict

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score
from sklearn.preprocessing import RobustScaler

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # go up from misc/ → realistic-al/

from src.models.networks.inception import Inception, InceptionBlock
from src.data.ecg5000_dataset import ECG5000Dataset

In [5]:
class Flatten(nn.Module):
	def __init__(self, out_features):
		super(Flatten, self).__init__()
		self.output_dim = out_features

	def forward(self, x):
		return x.view(-1, self.output_dim)
    
class Reshape(nn.Module):
	def __init__(self, out_shape):
		super(Reshape, self).__init__()
		self.out_shape = out_shape

	def forward(self, x):
		return x.view(-1, *self.out_shape)

In [11]:
X = np.vstack((np.load("/home/jing/Desktop/realistic-al/misc/InceptionTime-Pytorch-master/data/sequenced_data_for_VAE_length-160_stride-10_pt1.npy"),
               np.load("/home/jing/Desktop/realistic-al/misc/InceptionTime-Pytorch-master/data/sequenced_data_for_VAE_length-160_stride-10_pt2.npy")))
y = np.load("/home/jing/Desktop/realistic-al/misc/InceptionTime-Pytorch-master/data/sequenced_data_for_VAE_length-160_stride-10_targets.npy")

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=666)

In [13]:
X_train.shape

(66944, 160)

In [14]:
scaler = RobustScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [9]:
InceptionTime = nn.Sequential(
                    Reshape(out_shape=(1,160)),
                    InceptionBlock(
                        in_channels=1, 
                        n_filters=32, 
                        kernel_sizes=[5, 11, 23],
                        bottleneck_channels=32,
                        use_residual=True,
                        activation=nn.ReLU()
                    ),
                    InceptionBlock(
                        in_channels=32*4, 
                        n_filters=32, 
                        kernel_sizes=[5, 11, 23],
                        bottleneck_channels=32,
                        use_residual=True,
                        activation=nn.ReLU()
                    ),
                    nn.AdaptiveAvgPool1d(output_size=1),
                    Flatten(out_features=32*4*1),
                    nn.Linear(in_features=4*32*1, out_features=4)
        )

In [17]:
sum([x.numel() for x in InceptionTime.parameters() if x.requires_grad])

261252

In [16]:
sum([x.numel() for x in resnet.parameters() if x.requires_grad])

11689512

In [12]:
InceptionTime.load_state_dict(torch.load("/home/jing/Desktop/realistic-al/misc/InceptionTime-Pytorch-master/InceptionTime_full-version_lr-{5e-3,1e-3,2e-4},_bs-512_ks-[5,11,23]_100-epochs_state_dict.pt"))

<All keys matched successfully>

In [13]:
InceptionTime

Sequential(
  (0): Reshape()
  (1): InceptionBlock(
    (activation): ReLU()
    (inception_1): Inception(
      (conv_from_bottleneck_1): Conv1d(1, 32, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (conv_from_bottleneck_2): Conv1d(1, 32, kernel_size=(11,), stride=(1,), padding=(5,), bias=False)
      (conv_from_bottleneck_3): Conv1d(1, 32, kernel_size=(23,), stride=(1,), padding=(11,), bias=False)
      (max_pool): MaxPool1d(kernel_size=3, stride=1, padding=1, dilation=1, ceil_mode=False)
      (conv_from_maxpool): Conv1d(1, 32, kernel_size=(1,), stride=(1,), bias=False)
      (batch_norm): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (activation): ReLU()
    )
    (inception_2): Inception(
      (bottleneck): Conv1d(128, 32, kernel_size=(1,), stride=(1,), bias=False)
      (conv_from_bottleneck_1): Conv1d(32, 32, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (conv_from_bottleneck_2): Conv1d(32, 32, kernel_siz

In [14]:
InceptionTime.eval()
with torch.no_grad():
    x_pred = np.argmax(InceptionTime(torch.tensor(X_test).float()).detach(), axis=1)
x_pred

tensor([1, 1, 1,  ..., 3, 1, 0])

In [15]:
torch.tensor(X_test).float().shape

torch.Size([16736, 160])

In [15]:
f1_score(y_true=y_test, y_pred=x_pred,average="macro")

0.9125625555928921

In [16]:
accuracy_score(y_true=y_test, y_pred=x_pred)

0.9493307839388145

In [17]:
cf1 = confusion_matrix(y_true=y_test, y_pred=x_pred) # x_axis = predicted, y_axis = ground_truth
cf1

array([[ 3805,   296,    27,   107],
       [   91, 10307,     8,    39],
       [   22,     8,   425,    24],
       [  169,    37,    20,  1351]])

In [26]:
from torch.utils.data import Dataset, DataLoader
import scipy.io.arff as arff 
import pandas as pd
class ECG5000(Dataset):

    def __init__(self, mode):

        assert mode in ['normal', 'anomaly']

        trainset_file = '/home/jing/Desktop/realistic-al/datasets/ecg5000/ECG5000_TRAIN.arff'
        testset_file ='/home/jing/Desktop/realistic-al/datasets/ecg5000/ECG5000_TEST.arff'

        traindata, trainmeta = arff.loadarff(trainset_file)
        testdata, testmeta = arff.loadarff(testset_file)
        train = pd.DataFrame(traindata, columns=trainmeta.names())
        test = pd.DataFrame(testdata, columns=testmeta.names())
        # df = train.append(test)
        df = pd.concat([train, test])

        # split in normal and anomaly data, then drop label
        CLASS_NORMAL = 1
        new_columns = list(df.columns)
        new_columns[-1] = 'target'
        df.columns = new_columns


        if mode == 'normal':
            df = df[df.target == b'1'].drop(labels='target', axis=1)
        else:
            df = df[df.target != b'1'].drop(labels='target', axis=1)

        print(df.shape)
        # train_df, val_df = train_test_split(
        #     normal_df,
        #     test_size=0.15,
        #     random_state=random_seed
        # )
        #
        # val_df, test_df = train_test_split(
        #     val_df,
        #     test_size=0.33,
        #     random_state=random_seed
        # )

        self.X = df.astype(np.float32).to_numpy()

    def get_torch_tensor(self):
        return torch.from_numpy(self.X)

    def __getitem__(self, index):
        return torch.from_numpy(self.X[index]).reshape(-1, 1)

    # return len of dataset
    def __len__(self):
        return self.X.shape[0]

In [27]:
ecg5000 = ECG5000(mode='anomaly')

(2081, 140)


In [5]:
test_dataset = ECG5000Dataset(split='test')
train_dataset = ECG5000Dataset(split='train')

In [6]:
len(train_dataset), len(test_dataset)

(500, 4500)

In [10]:
loader = DataLoader(train_dataset, batch_size=10)

x, y = next(iter(loader))
print(x.shape)
print(y.shape)

torch.Size([10, 1, 140])
torch.Size([10])


In [16]:
import torch.nn as nn
from src.models.networks.inception import InceptionBlock

model = nn.Sequential(
    InceptionBlock(in_channels=1,   n_filters=32, bottleneck_channels=32, use_residual=True),
    InceptionBlock(in_channels=128, n_filters=32, bottleneck_channels=32, use_residual=True),
    nn.AdaptiveAvgPool1d(1),   # (B, 128, T) → (B, 128, 1)
    nn.Flatten(),              # (B, 128, 1) → (B, 128)
    nn.Linear(128, 5),         # 5 ECG classes
)

# Sanity check: feed a fake batch through
import torch
x = torch.randn(4, 1, 140)    # batch of 4
print(model(x).shape)         # → torch.Size([4, 5])


torch.Size([4, 5])
